# TAEL V2 (`v2.0.0-rc1`) — Sandboxed Tools, Persistent Memory & Autonomous Agent Loop

This notebook demonstrates:
1. **Sandboxed Tool Registry (`tael_tools`)**: Executing workspace tools (`list_dir`, `search_text`, `read_file`, `execute_python`) and verifying `PathSandbox` / `CommandSandbox` security guards.
2. **4-Pillar Persistent Memory (`tael_memory`)**: Tracking files, architectural decisions, rolling conversation summaries, and enforcing the `4,096`-token `ContextBudget`.
3. **Fault-Tolerant XML `<tool_call>` Parsing (`XMLToolParser`)**.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from tael_tools import create_tael_tool_registry

registry = create_tael_tool_registry(workspace_dir=str(Path('..').resolve()))
print("Registered Tools (11):", registry.list_tool_names())

print("\n--- Executing `execute_python` ---")
res = registry.execute("execute_python", {"code": "fib = lambda n: n if n < 2 else fib(n-1)+fib(n-2); print([fib(i) for i in range(10)])"})
print("Status:", res.status)
print("Output:", res.output.strip())

print("\n--- Testing PathSandbox Security Guard ---")
blocked = registry.execute("read_file", {"path": "../../Windows/System32/cmd.exe"})
print("Blocked traversal status:", blocked.status)
print("Message:", blocked.error)

Registered Tools (11): ['read_file', 'write_file', 'list_dir', 'search_text', 'git_status', 'git_diff', 'git_log', 'git_branch', 'execute_python', 'execute_terminal_command', 'finish']

--- Executing `execute_python` ---
Status: success
Output: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

--- Testing PathSandbox Security Guard ---
Blocked traversal status: error
Message: Access denied: Path '../../Windows/System32' resolves outside the allowed workspace sandbox.


## 4-Pillar Persistent Memory & `ContextBudget` Verification

In [2]:
import tempfile
from tael_memory import MemoryManager, ContextBudget

with tempfile.TemporaryDirectory() as tmpdir:
    mem = MemoryManager(workspace_dir=tmpdir, budget=ContextBudget(max_context=4096))
    mem.project.add_decision(
        title="Use SQLite WAL Mode",
        decision="Enable PRAGMA journal_mode=WAL for concurrent local reads.",
        rationale="Prevents database locked errors during multi-agent tool calls.",
        tags=["sqlite", "database", "concurrency"]
    )
    mem.record_user_message("How should we configure SQLite concurrency in our database module?")
    
    messages = mem.prepare_messages_for_inference(
        base_system_prompt="You are TAEL, an autonomous software engineering assistant.",
        current_query="How should we configure SQLite concurrency in our database module?"
    )
    est_tokens = sum(len(m['content']) // 4 for m in messages)
    print(f"Prepared messages count: {len(messages)}")
    print(f"Estimated Input Tokens: {est_tokens} / {mem.budget.max_input_tokens} max budget")
    print(f"Within 4096 Token Context Window: {est_tokens <= mem.budget.max_input_tokens}")

Prepared messages count: 3
Estimated Input Tokens: 214 / 2998 max budget
Within 4096 Token Context Window: True
